# Context Window

**Tags:** model-architecture, capacity
**Prerequisites:** None
**Related Concepts:** See flowchart below
**Source:** llm/concepts/context-window.md

## TL;DR

Maximum input tokens an LLM can process in one forward pass. Ranges: 4K → 32K → 128K → 200K+ (Claude 3.5 Sonnet). Larger window enables processing full documents but with O(T²) cost: 2x context = ~2-4x latency, ~2x cost. Position encoding and attention complexity drive trade-offs.

## Core Intuition

Transformers use all-to-all attention: every token attends to every other token, creating a T×T attention matrix. This is O(T²) memory and compute. Larger context windows mean bigger matrices, slower inference, higher cost. But they also enable handling whole documents in one pass, avoiding context chunking.

## How It Works

**Attention Complexity with Context:**
```
Sequence length: T tokens
Attention matrix: Q @ K^T → shape (T, T)
Memory per sequence: O(T²) × (dtype size)
Compute: O(T² × D) where D = hidden dimension

Example (GPT-4):
  T=4K (4,096 tokens): attention matrix = 4K × 4K × 4 bytes = 64 MB (moderate)
  T=128K (131K tokens): attention matrix = 128K × 128K × 4 bytes = 64 GB (massive!)
```

**Latency Scaling:**
```
Inference latency ≈ O(T²/parallel_dims)

For T=4K on H100: ~100ms per token
For T=32K on H100: ~500ms per token (roughly 5x)
For T=128K on H100: ~2s per token (20x slower)

Token costs:
  GPT-4 8K: $0.03 per 1K tokens
  GPT-4 32K: $0.06 per 1K tokens (2x cost)
  GPT-4 128K: $0.12 per 1K tokens (4x cost)
```

**Typical Window Sizes & Use Cases:**

| Window | Tokens | Words | Pages | Use Case |
|--------|--------|-------|-------|----------|
| 4K | 4,096 | ~3,000 | 3-4 | Chat, short QA |
| 8K | 8,192 | ~6,000 | 6-8 | Document analysis |
| 32K | 32,768 | ~24,000 | 24-30 | Book chapter, code review |
| 128K | 131,072 | ~96,000 | 100+ | Full books, repositories |
| 200K+ | 200K-1M | ~150K-750K | 150-750 | Research docs, large codebases |

**Position Encoding & Scaling:**

Modern LLMs trained on specific context windows:
```
GPT-3/3.5: 4K context, position encoding for 0-4096
Giving it 32K: positions beyond training range
  → accuracy drops (models haven't learned position patterns for token 8000+)
  → extrapolation fails

Solutions:
1. Positional Interpolation: compress positions to fit training range
   pos_interpolated = pos × (training_window / target_window)
   
2. Rotary Embeddings (RoPE): can extrapolate better than absolute positions
   
3. Fine-tuning on longer sequences: re-train position encoding on longer contexts
```

### Workflow Diagram

```mermaid
graph LR
    A["Input"] --> B["Context Window Process"]
    B --> C["Output"]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#e8f5e9
```

**Note:** This is a basic workflow template. Review and customize based on specific concept.

## Key Properties & Trade-offs

/ Trade-offs

| Property | 4K | 32K | 128K | 200K |
|----------|-----|--------|--------|---------|
| Latency | Fast (baseline) | 5-10x | 20-50x | 50-100x |
| Memory | 64 MB | 4 GB | 64 GB | 160 GB |
| Cost | $0.03/1K | $0.06/1K | $0.12/1K | $0.15/1K |
| Chunking needed | Yes | Maybe | No | No |
| Quality | High | High | High* | Medium** |

*Higher quality when content fits; **some quality loss at extremes due to position encoding

**Real-world implications:**
```
Task: Analyze 100-page report (300K words)

Approach 1: 4K context
- Chunk into 25 × 4K chunks
- Analyze each separately
- Cost: 25 × $0.03 = $0.75
- Problem: loses cross-chunk context

Approach 2: 128K context
- Fits entire report in 4 passes
- Cost: 4 × $0.12 = $0.48
- Benefit: maintains full document context

Approach 3: 200K context
- Fits entire report in 2 passes
- Cost: 2 × $0.15 = $0.30
- Benefit: best context, cheapest overall
```

### Code Implementation

```python
# Key imports for Context Window
import numpy as np
import torch
from typing import Any

# Context Window example implementation
class ContextWindow:
    """
    Context Window implementation.
    This is a template - customize with actual code.
    """
    def __init__(self):
        pass

    def process(self, input_data: Any) -> Any:
        # Interview tip: Explain the core insight here
        return input_data
```

### Mathematical Formulation

Include LaTeX equations relevant to this concept.

**Example:**
$$\text{Output} = f(\text{Input})$$

## Related Concepts

```mermaid
graph TD
    A["Context Window"]
    A -->|used with| D["Token Optimization"]
    
    style A fill:#fff3e0
```

### Common Interview Questions

**Q: What is Context Window used for?**
A: [Add concise answer about practical application]

**Q: What are the main trade-offs of Context Window?**
A: [Discuss pros/cons and when to use vs alternatives]

**Q: How does Context Window compare to [related concept]?**
A: [Explain key differences and when to use each]

**Q: What are common mistakes when using Context Window?**
A: [List 1-2 common pitfalls and how to avoid them]

**Q: Can you explain the intuition behind Context Window?**
A: [Provide a simple analogy or explanation]

## References

- **Source Document:** `llm/concepts/context-window.md`
- **Related Papers:** [Add relevant papers]
- **Implementations:**
  - HuggingFace: [Add links]
  - GitHub: [Add links]